# C5061 | Experimentos reproducibles con MLflow
**Marco Emilio Jimenez Jimenez · A01797948**  
Operaciones de aprendizaje automático · Tecnológico de Monterrey

Este notebook prepara y ejecuta `train.py` en **Google Colab**. El problema sigue siendo
estimar la calidad del vino tinto mediante regresión. El entrenamiento, el preprocesamiento
y la evaluación están en funciones del script; aquí solo se prepara el entorno, se lanzan
comandos y se revisan los resultados registrados.

**Lo que se conserva del baseline:** CSV Wine Quality de UCI, 11 predictores, objetivo `quality`,
limpieza numérica, eliminación de duplicados exactos, imputación por mediana, `RobustScaler`,
`LinearRegression`, 20 % de prueba y semilla 42. Los valores extremos se conservan.

**Cambio necesario:** `LinearRegression` tiene pocos hiperparámetros útiles para este ejercicio.
Se registra el baseline y se compara con `ElasticNet`, que sigue siendo regresión lineal,
pero agrega regularización. Sus tres hiperparámetros son `alpha`, `l1_ratio` y `fit_intercept`.
La semilla es un control de reproducibilidad, no uno de esos tres hiperparámetros.
Se toma validación del conjunto de desarrollo: aproximadamente 60 % entrenamiento,
20 % validación y 20 % prueba.

In [1]:
from pathlib import Path
import os

PROYECTO = Path('/content/c5061_mlflow')
PROYECTO.mkdir(parents=True, exist_ok=True)
os.chdir(PROYECTO)
print('Carpeta de trabajo:', Path.cwd())


Carpeta de trabajo: /content/c5061_mlflow


## Instalar versiones fijas
Esta celda escribe `requirements.txt`. `argparse`, `random`, `json`, `hashlib`, `subprocess`
y `pathlib` pertenecen a Python y no se instalan con pip. Colab ya proporciona IPython y
`google.colab`. El CSV se descarga desde UCI en la primera ejecución.


In [3]:
import subprocess
import sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)


CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], returncode=0)

In [4]:
import importlib.metadata as metadata
import json
import platform

versiones = {}
for linea in Path('requirements.txt').read_text().splitlines():
    paquete, esperada = linea.split('==')
    actual = metadata.version(paquete)
    assert actual == esperada, f'{paquete}: {actual} != {esperada}'
    versiones[paquete] = actual
entorno = {'python': sys.version, 'plataforma': platform.platform(), 'paquetes': versiones}
Path('entorno.json').write_text(json.dumps(entorno, indent=2), encoding='utf-8')
lock = subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True)
Path('requirements-lock.txt').write_text(lock, encoding='utf-8')
print(json.dumps(entorno, indent=2))


{
  "python": "3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]",
  "plataforma": "Linux-6.6.122+-x86_64-with-glibc2.39",
  "paquetes": {
    "numpy": "2.2.6",
    "pandas": "2.2.3",
    "scipy": "1.15.3",
    "scikit-learn": "1.6.1",
    "matplotlib": "3.10.3",
    "mlflow": "3.16.1",
    "cloudpickle": "3.1.1"
  }
}


El archivo fijo contiene las dependencias directas; `requirements-lock.txt` captura también
las dependencias transitivas del entorno utilizado. El lock de Colab incluye paquetes del
servicio y sirve como evidencia, mientras que `requirements.txt` es la instalación recomendada.
Dos corridas se comparan en el mismo entorno y con el mismo CSV y código. Se esperan métricas
y predicciones iguales dentro de una tolerancia numérica de `1e-10`; el identificador de corrida,
el tiempo de ejecución y los archivos del modelo pueden variar. No se promete igualdad de bytes
entre diferentes sistemas operativos o versiones de Python.


## 3. Crear el script de entrenamiento
Ejecuta la celda `%%writefile`: guarda un archivo Python completo. La lógica de entrenamiento
se escribe una sola vez y después se ejecuta como proceso independiente.

| Función | Responsabilidad |
|---|---|
| `argumentos` | Leer y validar argumentos, mostrar ayuda y valores por defecto |
| `cargar_datos` | Descargar o leer CSV, comprobar columnas, limpiar y calcular hash |
| `dividir_datos` | Crear siempre las mismas tres particiones con la misma semilla |
| `crear_modelo` | Construir imputación, escalado y regresor dentro de un pipeline |
| `evaluar` | Calcular MAE, RMSE y R² |
| `guardar_artefactos` | Guardar métricas, predicciones, índices y gráfica |
| `main` | Fijar semilla, entrenar, registrar y guardar el resumen de una corrida |

La semilla controla `random`, NumPy, las dos particiones y `ElasticNet`. Se fija un hilo numérico
antes de importar NumPy. `LinearRegression` y el solver cíclico de ElasticNet son deterministas
para datos y entorno iguales. El pipeline ajusta la mediana y el escalado solo con entrenamiento.


In [5]:
%%writefile train.py
"""Entrenamiento reproducible de regresión para Wine Quality."""
import os

# Se fijan antes de importar las bibliotecas numéricas.
for variable in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ[variable] = "1"

import argparse
import hashlib
import json
import random
import sys
import tempfile
from pathlib import Path
from urllib.request import urlretrieve

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
from sklearn.dummy import DummyRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import ElasticNet, LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import RobustScaler

URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv"
FEATURES = ["fixed acidity", "volatile acidity", "citric acid", "residual sugar",
            "chlorides", "free sulfur dioxide", "total sulfur dioxide", "density",
            "pH", "sulphates", "alcohol"]


def argumentos():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--model", choices=["linear", "elasticnet"], default="linear",
                        help="linear conserva el modelo original; elasticnet añade regularización")
    parser.add_argument("--alpha", type=float, default=0.01,
                        help="Intensidad de regularización de ElasticNet, mayor que cero (0.01)")
    parser.add_argument("--l1-ratio", type=float, default=0.5,
                        help="Proporción L1 de ElasticNet entre 0 y 1 (0.5)")
    parser.add_argument("--fit-intercept", type=int, choices=[0, 1], default=1,
                        help="Estimar intercepto: 1 sí, 0 no (1)")
    parser.add_argument("--seed", type=int, default=42, help="Semilla de las particiones y modelo (42)")
    parser.add_argument("--data", default="winequality-red.csv", help="CSV separado por punto y coma")
    parser.add_argument("--experiment", default="C5061_vino_regresion_parametrizada",
                        help="Nombre del experimento de MLflow")
    parser.add_argument("--tracking-uri", default="sqlite:///mlflow.db", help="Base de MLflow")
    parser.add_argument("--run-name", default="baseline_lineal", help="Nombre visible de la corrida")
    parser.add_argument("--stage", choices=["comparacion", "reproducibilidad", "final"],
                        default="comparacion", help="Propósito de la corrida")
    parser.add_argument("--evaluate-test", action="store_true", help="Evaluar prueba solo para el ganador")
    parser.add_argument("--result-file", default="ultima_corrida.json", help="Resumen de salida")
    args = parser.parse_args()
    if not np.isfinite(args.alpha) or args.alpha <= 0:
        parser.error("alpha debe ser finito y mayor que cero")
    if not 0 <= args.l1_ratio <= 1:
        parser.error("l1-ratio debe estar entre 0 y 1")
    if not 0 <= args.seed < 2**32:
        parser.error("seed debe estar entre 0 y 2**32 - 1")
    if args.evaluate_test != (args.stage == "final"):
        parser.error("La etapa final requiere --evaluate-test; las otras etapas no lo permiten")
    return args


def cargar_datos(ruta):
    archivo = Path(ruta)
    if not archivo.exists():
        archivo.parent.mkdir(parents=True, exist_ok=True)
        urlretrieve(URL, archivo)
    original = pd.read_csv(archivo, sep=";")
    if set(original.columns) != set(FEATURES + ["quality"]):
        raise ValueError("El CSV no tiene las 11 variables y quality esperadas")
    datos = original.apply(pd.to_numeric, errors="coerce")
    datos = datos.dropna(subset=["quality"]).drop_duplicates().reset_index(drop=True)
    datos["quality"] = datos["quality"].astype(int)
    return datos, {"sha256_csv": hashlib.sha256(archivo.read_bytes()).hexdigest(),
                   "filas_originales": len(original), "filas_limpias": len(datos),
                   "filas_eliminadas": len(original) - len(datos)}


def dividir_datos(datos, seed):
    X, y = datos[FEATURES], datos["quality"]
    # Se conserva el 20 % de prueba y seed=42 del notebook original.
    X_dev, X_test, y_dev, y_test = train_test_split(X, y, test_size=0.20, random_state=seed)
    # 25 % del 80 % = 20 % del total para validación; queda 60 % para entrenar.
    X_train, X_val, y_train, y_val = train_test_split(
        X_dev, y_dev, test_size=0.25, random_state=seed)
    return X_train, X_val, X_test, y_train, y_val, y_test


def crear_modelo(args):
    if args.model == "linear":
        regresor = LinearRegression(fit_intercept=bool(args.fit_intercept))
    else:
        regresor = ElasticNet(alpha=args.alpha, l1_ratio=args.l1_ratio,
                             fit_intercept=bool(args.fit_intercept), random_state=args.seed,
                             selection="cyclic", max_iter=10000, tol=1e-7)
    # Mediana y escalado se aprenden exclusivamente con entrenamiento.
    return make_pipeline(SimpleImputer(strategy="median"), RobustScaler(), regresor)


def evaluar(y, pred, prefijo):
    return {f"mae_{prefijo}": float(mean_absolute_error(y, pred)),
            f"rmse_{prefijo}": float(np.sqrt(mean_squared_error(y, pred))),
            f"r2_{prefijo}": float(r2_score(y, pred))}


def guardar_artefactos(carpeta, metricas, y_val, pred, particiones):
    (carpeta / "metricas.json").write_text(json.dumps(metricas, indent=2), encoding="utf-8")
    pd.DataFrame({"indice": y_val.index, "real": y_val.to_numpy(),
                  "prediccion": pred}).to_csv(carpeta / "predicciones_validacion.csv", index=False)
    (carpeta / "particiones.json").write_text(json.dumps(particiones, indent=2), encoding="utf-8")
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.scatter(y_val, pred, alpha=0.4)
    ax.plot([y_val.min(), y_val.max()], [y_val.min(), y_val.max()], "r--")
    ax.set(xlabel="Calidad real", ylabel="Calidad estimada", title="Predicciones en validación")
    fig.tight_layout()
    fig.savefig(carpeta / "predicciones.png", dpi=150)
    plt.close(fig)


def main():
    args = argumentos()
    random.seed(args.seed)
    np.random.seed(args.seed)
    mlflow.set_tracking_uri(args.tracking_uri)
    experimento = mlflow.set_experiment(args.experiment)
    datos, informacion = cargar_datos(args.data)
    X_train, X_val, X_test, y_train, y_val, y_test = dividir_datos(datos, args.seed)
    particiones = {"train": X_train.index.tolist(), "validation": X_val.index.tolist(),
                   "test": X_test.index.tolist()}
    modelo = crear_modelo(args)
    with mlflow.start_run(run_name=args.run_name) as run:
        parametros = {"model": args.model, "fit_intercept": args.fit_intercept, "seed": args.seed,
                      "test_size": 0.20, "validation_fraction_dev": 0.25,
                      "imputacion": "median", "escalado": "RobustScaler",
                      "n_train": len(X_train), "n_val": len(X_val), "n_test": len(X_test),
                      **informacion}
        if args.model == "elasticnet":
            parametros.update(alpha=args.alpha, l1_ratio=args.l1_ratio, max_iter=10000,
                              tol=1e-7, selection="cyclic")
        for clave, valor in parametros.items():
            mlflow.log_param(clave, valor)
        mlflow.set_tags({"etapa": args.stage, "sha256_script": hashlib.sha256(
            Path(__file__).read_bytes()).hexdigest(), "python_version": sys.version.split()[0]})
        modelo.fit(X_train, y_train)
        pred = modelo.predict(X_val)
        metricas = evaluar(y_val, pred, "val")
        metricas.update(evaluar(y_train, modelo.predict(X_train), "train"))
        ingenuo = DummyRegressor(strategy="mean").fit(X_train, y_train)
        metricas["mae_media_val"] = float(mean_absolute_error(y_val, ingenuo.predict(X_val)))
        if args.evaluate_test:
            metricas.update(evaluar(y_test, modelo.predict(X_test), "test"))
        for clave, valor in metricas.items():
            mlflow.log_metric(clave, valor)
        with tempfile.TemporaryDirectory() as temporal:
            carpeta = Path(temporal)
            guardar_artefactos(carpeta, metricas, y_val, pred, particiones)
            for archivo in carpeta.iterdir():
                mlflow.log_artifact(str(archivo), artifact_path="evaluacion")
        for archivo in [Path(__file__), Path(args.data), Path("requirements.txt"),
                        Path("requirements-lock.txt"), Path("entorno.json")]:
            if archivo.exists():
                mlflow.log_artifact(str(archivo), artifact_path="reproducibilidad")
        info_modelo = mlflow.sklearn.log_model(
            modelo, name="modelo", input_example=X_train.head(3),
            serialization_format="cloudpickle", pip_requirements="requirements.txt")
        resumen = {"run_id": run.info.run_id, "experiment_id": experimento.experiment_id,
                   "run_name": args.run_name, "model_uri": info_modelo.model_uri,
                   "parametros": parametros, "metricas": metricas}
    destino = Path(args.result_file)
    destino.parent.mkdir(parents=True, exist_ok=True)
    destino.write_text(json.dumps(resumen, indent=2), encoding="utf-8")
    print(json.dumps(resumen, indent=2))


if __name__ == "__main__":
    main()


Writing train.py


## Probar los valores por defecto
Ejecuta la ayuda y verifica que aparezcan `--alpha`, `--l1-ratio`, `--fit-intercept` y `--seed`.
`python train.py` funciona sin argumentos y conserva la regresión lineal del modelo base.
`alpha` y `l1_ratio` solo se aplican y registran cuando se elige `--model elasticnet`.
Para evitar confundir valores inactivos, el baseline no registra esos dos parámetros.

Cada corrida registra parámetros con `log_param`, MAE/RMSE/R² con `log_metric`, archivos
con `log_artifact` y el pipeline completo con `mlflow.sklearn.log_model`, dentro de `start_run`.
No se necesita autologging: el registro explícito permite comprobar cada requisito.


In [6]:
subprocess.run([sys.executable, 'train.py', '--help'], check=True)


CompletedProcess(args=['/usr/bin/python3', 'train.py', '--help'], returncode=0)

In [7]:
# Sin argumentos: primera corrida, modelo original y semilla 42.
subprocess.run([sys.executable, 'train.py'], check=True)
baseline = json.loads(Path('ultima_corrida.json').read_text())
baseline['nombre'] = 'B00_lineal'
print('Baseline registrado:', baseline['run_id'])


Baseline registrado: b4333c1b66bd48049fcb7272daa8e4e3


## Ejecutar seis configuraciones adicionales
El experimento se llama **C5061_vino_regresion_parametrizada**. Se comparan siete configuraciones
distintas contando el baseline. Todas usan el mismo CSV, semilla 42, preprocesamiento y particiones.
**Menor MAE de validación** es el criterio de selección definido antes de observar resultados.
RMSE y R² complementan la interpretación; RMSE menor y R² mayor son preferibles.

| Nombre | Modelo | alpha | l1_ratio | fit_intercept |
|---|---|---:|---:|---:|
| B00 | LinearRegression | No aplica | No aplica | 1 |
| E01 | ElasticNet | 0.001 | 0.5 | 1 |
| E02 | ElasticNet | 0.01 | 0.5 | 1 |
| E03 | ElasticNet | 0.1 | 0.5 | 1 |
| E04 | ElasticNet | 0.01 | 0.1 | 1 |
| E05 | ElasticNet | 0.01 | 0.9 | 1 |
| E06 | ElasticNet | 0.01 | 0.5 | 0 |

E02 es el punto de referencia: E01/E03 cambian únicamente alpha; E04/E05 cambian únicamente
l1_ratio; E06 cambia únicamente el intercepto. Así se puede argumentar influencia local sin
cambiar simultáneamente todos los factores. `fit_intercept=0` obliga a pasar por el origen
en el espacio escalado y puede resultar inadecuado para puntuaciones cercanas a 5 o 6.


In [8]:
EXPERIMENTO = 'C5061_vino_regresion_parametrizada'
configuraciones = [
    ('E01_alpha_0001', 0.001, 0.5, 1),
    ('E02_referencia', 0.01, 0.5, 1),
    ('E03_alpha_01', 0.1, 0.5, 1),
    ('E04_l1_01', 0.01, 0.1, 1),
    ('E05_l1_09', 0.01, 0.9, 1),
    ('E06_sin_intercepto', 0.01, 0.5, 0),
]
resultados = [baseline]
for nombre, alpha, l1, intercepto in configuraciones:
    archivo = f'resumenes/{nombre}.json'
    comando = [sys.executable, 'train.py', '--model', 'elasticnet',
               '--alpha', str(alpha), '--l1-ratio', str(l1),
               '--fit-intercept', str(intercepto), '--seed', '42',
               '--experiment', EXPERIMENTO, '--run-name', nombre, '--result-file', archivo]
    subprocess.run(comando, check=True)
    resultado = json.loads(Path(archivo).read_text())
    resultado['nombre'] = nombre
    resultados.append(resultado)
Path('corridas_comparacion.json').write_text(json.dumps(resultados, indent=2), encoding='utf-8')
print('Configuraciones distintas:', len(resultados))


Configuraciones distintas: 7


## Consultar la evidencia registrada y elegir el ganador
Se recuperan las métricas desde MLflow, no desde números escritos manualmente.
Se filtran los identificadores de este lote: reejecutar celdas puede generar otras corridas
del mismo experimento. Esas corridas anteriores no deben mezclarse con esta comparación.
En empates exactos de MAE, se usa RMSE y después el nombre para ordenar de forma estable.


In [1]:
from pathlib import Path
import os
import json
import sys
import subprocess

PROYECTO = Path("/content/c5061_mlflow")
os.chdir(PROYECTO)

import numpy as np
import pandas as pd
import scipy

print("NumPy:", np.__version__)
print("pandas:", pd.__version__)
print("SciPy:", scipy.__version__)

# Recuperar los resultados que ya quedaron guardados.
resultados = json.loads(
    Path("corridas_comparacion.json").read_text(encoding="utf-8")
)
EXPERIMENTO = "C5061_vino_regresion_parametrizada"

print("Configuraciones recuperadas:", len(resultados))

NumPy: 2.2.6
pandas: 2.2.3
SciPy: 1.15.3
Configuraciones recuperadas: 7


In [2]:
import mlflow
import mlflow.sklearn
import pandas as pd
import numpy as np
from IPython.display import display, Markdown, Image

mlflow.set_tracking_uri('sqlite:///mlflow.db')
filas = []
for resultado in resultados:
    corrida = mlflow.get_run(resultado['run_id'])
    p, m = corrida.data.params, corrida.data.metrics
    assert corrida.info.status == 'FINISHED'
    filas.append({'nombre': resultado['nombre'], 'run_id': resultado['run_id'],
                  'modelo': p['model'], 'alpha': float(p['alpha']) if 'alpha' in p else np.nan,
                  'l1_ratio': float(p['l1_ratio']) if 'l1_ratio' in p else np.nan,
                  'fit_intercept': int(p['fit_intercept']), 'seed': int(p['seed']),
                  'mae_val': m['mae_val'], 'rmse_val': m['rmse_val'], 'r2_val': m['r2_val'],
                  'mae_train': m['mae_train'], 'mae_media_val': m['mae_media_val']})
tabla = pd.DataFrame(filas).sort_values(['mae_val', 'rmse_val', 'nombre']).reset_index(drop=True)
tabla.to_csv('comparacion.csv', index=False)
display(tabla.round(6))
ganador = tabla.iloc[0]
print('Ganador por MAE de validación:', ganador['nombre'], ganador['run_id'])


,nombre,run_id,modelo,alpha,l1_ratio,fit_intercept,seed,mae_val,rmse_val,r2_val,mae_train,mae_media_val
0,E04_l1_01,41900660a3a745f39bf82d5206090f8c,elasticnet,0.010,0.1,1,42,0.536300,0.678628,0.362408,0.502337,0.710551
1,E02_referencia,09a21e23fe1c427ead59eac662645874,elasticnet,0.010,0.5,1,42,0.536535,0.677264,0.364968,0.503600,0.710551
2,E01_alpha_0001,006de08bae8f479d978278879a232117,elasticnet,0.001,0.5,1,42,0.536694,0.678968,0.361769,0.501777,0.710551
3,E05_l1_09,c67befab1a7441879ee1f676b7f9350e,elasticnet,0.010,0.9,1,42,0.536937,0.676862,0.365721,0.505757,0.710551
4,B00_lineal,b4333c1b66bd48049fcb7272daa8e4e3,linear,NaN,NaN,1,42,0.536972,0.679170,0.361390,0.501665,0.710551
5,E03_alpha_01,c7d31d8372af4d47bcd8a504e44fa954,elasticnet,0.100,0.5,1,42,0.556081,0.696984,0.327450,0.533527,0.710551
6,E06_sin_intercepto,b51921d6028648de9638b400334af67a,elasticnet,0.010,0.5,0,42,3.598558,4.246171,-23.961716,3.667117,0.710551


Ganador por MAE de validación: E04_l1_01 41900660a3a745f39bf82d5206090f8c


## Comprobar reproducibilidad con dos procesos independientes
Se repite E02 dos veces con exactamente los mismos argumentos de modelado, solo cambia el
nombre de corrida y el archivo de salida. Las repeticiones se etiquetan `reproducibilidad`
y no cuentan como configuraciones nuevas. Se comparan todas las métricas, las predicciones
fila por fila, el hash del CSV, el hash del script y las particiones guardadas.


In [3]:
repeticiones = []
for i in [1, 2]:
    archivo = f'resumenes/repeticion_{i}.json'
    subprocess.run([sys.executable, 'train.py', '--model', 'elasticnet',
                    '--alpha', '0.01', '--l1-ratio', '0.5', '--fit-intercept', '1',
                    '--seed', '42', '--stage', 'reproducibilidad',
                    '--run-name', f'repeticion_{i}', '--result-file', archivo], check=True)
    repeticiones.append(json.loads(Path(archivo).read_text()))
a, b = repeticiones
assert a['parametros'] == b['parametros']
assert mlflow.get_run(a['run_id']).data.tags['sha256_script'] == mlflow.get_run(b['run_id']).data.tags['sha256_script']
for metrica in a['metricas']:
    np.testing.assert_allclose(a['metricas'][metrica], b['metricas'][metrica], rtol=0, atol=1e-10)
predicciones = []
indices = []
for r in repeticiones:
    ruta = mlflow.artifacts.download_artifacts(
        artifact_uri=f"runs:/{r['run_id']}/evaluacion/predicciones_validacion.csv")
    predicciones.append(pd.read_csv(ruta))
    ruta_indices = mlflow.artifacts.download_artifacts(
        artifact_uri=f"runs:/{r['run_id']}/evaluacion/particiones.json")
    indices.append(json.loads(Path(ruta_indices).read_text()))
assert indices[0] == indices[1]
np.testing.assert_allclose(predicciones[0], predicciones[1], rtol=0, atol=1e-10)
mensaje = 'Reproducibilidad comprobada: métricas, predicciones, datos, código y particiones coinciden.'
Path('reproducibilidad.txt').write_text(mensaje, encoding='utf-8')
print(mensaje)


Reproducibilidad comprobada: métricas, predicciones, datos, código y particiones coinciden.


## Evaluar una vez el ganador en prueba y recuperar su modelo
El ganador ya fue seleccionado con validación. Se ejecuta su configuración en una corrida
final con el mismo conjunto de entrenamiento; no se vuelve a ajustar usando prueba.
Se comprueba que conserva su MAE de validación. La prueba se reporta como estimación final,
sin usarla para cambiar el ganador. No se deben repetir estos pasos buscando una semilla favorable.


In [4]:
comando = [sys.executable, 'train.py', '--model', ganador['modelo'],
           '--fit-intercept', str(int(ganador['fit_intercept'])), '--seed', '42',
           '--stage', 'final', '--evaluate-test', '--run-name', 'ganador_evaluacion_final',
           '--result-file', 'resumenes/final.json']
if ganador['modelo'] == 'elasticnet':
    comando += ['--alpha', str(ganador['alpha']), '--l1-ratio', str(ganador['l1_ratio'])]
subprocess.run(comando, check=True)
final = json.loads(Path('resumenes/final.json').read_text())
np.testing.assert_allclose(final['metricas']['mae_val'], ganador['mae_val'], rtol=0, atol=1e-10)
modelo_recuperado = mlflow.sklearn.load_model(final['model_uri'])
# Comprobar que el pipeline recuperado reproduce las predicciones de validación.
csv_val = mlflow.artifacts.download_artifacts(
    artifact_uri=f"runs:/{final['run_id']}/evaluacion/predicciones_validacion.csv")
pred_guardada = pd.read_csv(csv_val)
datos = pd.read_csv('winequality-red.csv', sep=';').apply(pd.to_numeric, errors='coerce')
datos = datos.dropna(subset=['quality']).drop_duplicates().reset_index(drop=True)
X_ejemplo = datos.drop(columns='quality').iloc[pred_guardada['indice'].astype(int)]
np.testing.assert_allclose(modelo_recuperado.predict(X_ejemplo), pred_guardada['prediccion'],
                           rtol=0, atol=1e-10)
print('Modelo recuperado y predicciones verificadas:', final['model_uri'])
print('Métricas finales:', final['metricas'])


Modelo recuperado y predicciones verificadas: models:/m-df7bad1267ff47d497e861309c25028b
Métricas finales: {'mae_val': 0.536299628687591, 'rmse_val': 0.6786280881230354, 'r2_val': 0.3624075770520945, 'mae_train': 0.5023371534279236, 'rmse_train': 0.6530285015367683, 'r2_train': 0.3452387993425662, 'mae_media_val': 0.7105512450378925, 'mae_test': 0.5058167687467122, 'rmse_test': 0.6580995050896382, 'r2_test': 0.3885912680175321}


### Versionar el modelo seleccionado

In [5]:
version = mlflow.register_model(final['model_uri'], 'C5061_calidad_vino')
cliente = mlflow.tracking.MlflowClient()
cliente.set_registered_model_alias('C5061_calidad_vino', 'seleccionado', version.version)
registro = {'nombre': version.name, 'version': version.version, 'run_id': final['run_id'],
            'model_uri': final['model_uri'], 'alias': 'seleccionado'}
Path('modelo_seleccionado.json').write_text(json.dumps(registro, indent=2), encoding='utf-8')
print(registro)


{'nombre': 'C5061_calidad_vino', 'version': 1, 'run_id': '871ecdaaec91430fa8fd1d871a0baa75', 'model_uri': 'models:/m-df7bad1267ff47d497e861309c25028b', 'alias': 'seleccionado'}


Successfully registered model 'C5061_calidad_vino'.
Created version '1' of model 'C5061_calidad_vino'.


## MLflow en Google Colab

In [9]:
from pathlib import Path
from urllib.parse import urlsplit
import os
import socket
import subprocess
import sys
import time
from google.colab import output

PROYECTO = Path("/content/c5061_mlflow")
os.chdir(PROYECTO)

# Detener el servidor anterior si sigue activo.
if "servidor" in globals() and servidor.poll() is None:
    servidor.terminate()
    servidor.wait(timeout=30)

# Otro puerto evita reutilizar el servidor sin corregir.
PUERTO = 5001

with socket.socket() as s:
    if s.connect_ex(("127.0.0.1", PUERTO)) == 0:
        raise RuntimeError(
            "El puerto 5001 está ocupado. Cambia PUERTO a 5002."
        )

# Autorizar el dominio exacto del proxy de esta sesión.
url_proxy = output.eval_js(
    f"google.colab.kernel.proxyPort({PUERTO})"
)
proxy = urlsplit(url_proxy)

hosts = (
    f"localhost,localhost:*,127.0.0.1,127.0.0.1:*,"
    f"{proxy.hostname},{proxy.hostname}:*"
)
hosts += ",m-s-kkb-usw4a2-ceamck1yt623.us-west4-a.c.codatalab-user-runtimes.internal:8007"
origen = f"{proxy.scheme}://{proxy.netloc}"

log_servidor = open("mlflow_server.log", "w")

servidor = subprocess.Popen(
    [
        sys.executable, "-m", "mlflow", "server",
        "--backend-store-uri", "sqlite:///mlflow.db",
        "--default-artifact-root", (PROYECTO / "mlruns").as_uri(),
        "--no-serve-artifacts",
        "--host", "127.0.0.1",
        "--port", str(PUERTO),
        "--workers", "1",
        "--allowed-hosts", hosts,
        "--cors-allowed-origins",
        f"https://colab.research.google.com,{origen}",
        "--x-frame-options", "NONE",
    ],
    stdout=log_servidor,
    stderr=subprocess.STDOUT,
)

# Esperar a que el servidor esté listo.
for _ in range(45):
    if servidor.poll() is not None:
        raise RuntimeError(Path("mlflow_server.log").read_text())

    with socket.socket() as s:
        if s.connect_ex(("127.0.0.1", PUERTO)) == 0:
            break

    time.sleep(1)
else:
    raise RuntimeError("El servidor no inició. Revisa mlflow_server.log.")

# Mostrar MLflow dentro del notebook.
output.serve_kernel_port_as_iframe(PUERTO, height="850")

<IPython.core.display.Javascript object>

## 11. Analizar los resultados

In [11]:
por_nombre = tabla.set_index('nombre')
grupos = {'alpha': ['E01_alpha_0001', 'E02_referencia', 'E03_alpha_01'],
          'l1_ratio': ['E04_l1_01', 'E02_referencia', 'E05_l1_09'],
          'fit_intercept': ['E02_referencia', 'E06_sin_intercepto']}
efectos = {p: float(por_nombre.loc[nombres, 'mae_val'].max() -
                    por_nombre.loc[nombres, 'mae_val'].min()) for p, nombres in grupos.items()}
influyente = max(efectos, key=efectos.get)
descripcion = f"modelo={ganador['modelo']}, fit_intercept={int(ganador['fit_intercept'])}, seed=42"
if ganador['modelo'] == 'elasticnet':
    descripcion += f", alpha={ganador['alpha']:g}, l1_ratio={ganador['l1_ratio']:g}"
detalle = ', '.join(f'{p}: {v:.6f}' for p, v in efectos.items())
reduccion = 100 * (ganador['mae_media_val'] - ganador['mae_val']) / ganador['mae_media_val']
analisis = f"""# Análisis de resultados

Comparé siete configuraciones en el experimento {EXPERIMENTO}, con el mismo CSV, semilla y
particiones. Seleccioné {ganador['nombre']} ({descripcion}) porque obtuvo el menor MAE de
validación: {ganador['mae_val']:.6f} puntos. Su RMSE fue {ganador['rmse_val']:.6f} y su R²
{ganador['r2_val']:.6f}. Frente al predictor de la media, el cambio relativo del MAE fue
{reduccion:.2f}% (un valor positivo representa reducción). La corrida seleccionada es
{ganador['run_id']}. La tabla y la vista Compare de MLflow acompañan esta conclusión.

El factor más influyente en los valores probados fue {influyente}. Medí su efecto como el
rango del MAE al cambiar un parámetro y conservar los otros: {detalle}. Esta comparación
es local; los intervalos y opciones probados son distintos y no permiten afirmar que ese
factor sea siempre el más importante. Quitar el intercepto fuerza el origen en el espacio
escalado, algo poco apropiado para una calidad con media alejada de cero. La regularización
de ElasticNet modifica los coeficientes, pero no garantiza mejorar la regresión original.

Después de elegir el ganador, la corrida final obtuvo MAE de prueba
{final['metricas']['mae_test']:.6f}, RMSE {final['metricas']['rmse_test']:.6f} y R²
{final['metricas']['r2_test']:.6f}, sin usar prueba para seleccionar. Dos procesos con los
mismos argumentos reprodujeron métricas y predicciones dentro de 1e-10.

Las limitaciones son una sola partición, una búsqueda pequeña sin interacciones entre
hiperparámetros y el predominio de puntuaciones intermedias. La eliminación de duplicados
puede retirar muestras reales. La regresión lineal supone relaciones que pueden ser
insuficientes; RobustScaler cambia la escala, pero no vuelve robusta la función de pérdida.
Estos resultados no demuestran generalización a vinos de otra región o año. Como siguiente
paso usaría validación cruzada en desarrollo y revisaría el error por puntuación, manteniendo
prueba reservada para una evaluación final.
"""
Path('analisis_resultados.md').write_text(analisis, encoding='utf-8')
Path('sensibilidad.json').write_text(json.dumps(efectos, indent=2), encoding='utf-8')
display(Markdown(analisis))
print('Palabras:', len(analisis.split()))


# Análisis de resultados

Comparé siete configuraciones en el experimento C5061_vino_regresion_parametrizada, con el mismo CSV, semilla y
particiones. Seleccioné E04_l1_01 (modelo=elasticnet, fit_intercept=1, seed=42, alpha=0.01, l1_ratio=0.1) porque obtuvo el menor MAE de
validación: 0.536300 puntos. Su RMSE fue 0.678628 y su R²
0.362408. Frente al predictor de la media, el cambio relativo del MAE fue
24.52% (un valor positivo representa reducción). La corrida seleccionada es
41900660a3a745f39bf82d5206090f8c. La tabla y la vista Compare de MLflow acompañan esta conclusión.

El factor más influyente en los valores probados fue fit_intercept. Medí su efecto como el
rango del MAE al cambiar un parámetro y conservar los otros: alpha: 0.019545, l1_ratio: 0.000638, fit_intercept: 3.062023. Esta comparación
es local; los intervalos y opciones probados son distintos y no permiten afirmar que ese
factor sea siempre el más importante. Quitar el intercepto fuerza el origen en el espacio
escalado, algo poco apropiado para una calidad con media alejada de cero. La regularización
de ElasticNet modifica los coeficientes, pero no garantiza mejorar la regresión original.

Después de elegir el ganador, la corrida final obtuvo MAE de prueba
0.505817, RMSE 0.658100 y R²
0.388591, sin usar prueba para seleccionar. Dos procesos con los
mismos argumentos reprodujeron métricas y predicciones dentro de 1e-10.

Las limitaciones son una sola partición, una búsqueda pequeña sin interacciones entre
hiperparámetros y el predominio de puntuaciones intermedias. La eliminación de duplicados
puede retirar muestras reales. La regresión lineal supone relaciones que pueden ser
insuficientes; RobustScaler cambia la escala, pero no vuelve robusta la función de pérdida.
Estos resultados no demuestran generalización a vinos de otra región o año. Como siguiente
paso usaría validación cruzada en desarrollo y revisaría el error por puntuación, manteniendo
prueba reservada para una evaluación final.


Palabras: 283


In [12]:
readme = r"""# C5061 — Experimentos de calidad del vino

## Google Colab
Abrir el notebook, usar Python con CPU y ejecutar en orden. Instalar requirements.txt antes
de importar bibliotecas. El notebook crea train.py; entrenamiento y evaluación viven en ese
script. Consultar la sección 10 para comparación desde MLflow y capturas.

## Línea de comandos (Python 3.12 recomendado)
Desde la carpeta de este proyecto:
python -m venv .venv
# Linux/macOS: source .venv/bin/activate
# Windows PowerShell: .\.venv\Scripts\Activate.ps1
python -m pip install -r requirements.txt
python train.py --help
python train.py
python train.py --model elasticnet --alpha 0.001 --l1-ratio 0.5 --fit-intercept 1 --seed 42 --run-name E01
python train.py --model elasticnet --alpha 0.01 --l1-ratio 0.5 --fit-intercept 1 --seed 42 --run-name E02
python train.py --model elasticnet --alpha 0.1 --l1-ratio 0.5 --fit-intercept 1 --seed 42 --run-name E03
python train.py --model elasticnet --alpha 0.01 --l1-ratio 0.1 --fit-intercept 1 --seed 42 --run-name E04
python train.py --model elasticnet --alpha 0.01 --l1-ratio 0.9 --fit-intercept 1 --seed 42 --run-name E05
python train.py --model elasticnet --alpha 0.01 --l1-ratio 0.5 --fit-intercept 0 --seed 42 --run-name E06
python -m mlflow server --backend-store-uri sqlite:///mlflow.db --host 127.0.0.1 --port 5000
Abrir http://127.0.0.1:5000. En Colab usar el enlace del proxy que crea el notebook.

## Parámetros
model=linear por defecto conserva el baseline. ElasticNet expone alpha=0.01 (intensidad de
regularización), l1-ratio=0.5 (mezcla L1/L2) y fit-intercept=1 (estimar intercepto).
seed=42 controla random, NumPy, particiones y ElasticNet. Alpha y l1-ratio no se aplican al
baseline. --data permite un CSV local; se verifica el esquema y se registra SHA-256.
Prueba=20%, validación=20%, entrenamiento=60% aproximadamente; no se ajusta el pipeline con
validación ni prueba. Elegir por MAE de validación. La etapa final usa --stage final
--evaluate-test con los hiperparámetros del ganador, y prueba no cambia la selección.

## Evidencia y versiones
requirements.txt fija dependencias directas, requirements-lock.txt guarda el entorno completo,
entorno.json documenta Python y sistema. Cada corrida registra hashes del CSV y script,
particiones, métricas, predicciones, gráfica y pipeline. No se espera el mismo run_id o tiempos.
La comprobación de dos procesos compara resultados con atol=1e-10, rtol=0.
modelo_seleccionado.json vincula versión del registro, alias y corrida final.
comparacion.csv, analisis_resultados.md y evidencias/ documentan la selección.

## Respaldo
Guardar mlflow.db y mlruns/ juntos. En Colab extraer en /content/c5061_mlflow, la ruta original.
Para otro sistema, reinstalar y reconstruir las corridas con el CSV conservado y estos comandos.
No editar rutas de SQLite manualmente. Cerrar Colab sin descargar puede perder el experimento.
"""
Path('README.md').write_text(readme, encoding='utf-8')
evidencias = [p for p in Path('evidencias').glob('*') if p.suffix.lower() in ['.png', '.jpg', '.jpeg']]
assert len(evidencias) >= 2, 'Faltan al menos dos capturas reales de MLflow. Súbelas en la sección 10.'
display(pd.DataFrame({'archivo': [str(p) for p in Path('.').rglob('*') if p.is_file()]}))


,archivo
0,winequality-red.csv
1,ultima_corrida.json
2,reproducibilidad.txt
3,entorno.json
4,mlflow_server.log
...,...
186,mlruns/1/b4333c1b66bd48049fcb7272daa8e4e3/arti...
187,mlruns/1/b4333c1b66bd48049fcb7272daa8e4e3/arti...
188,mlruns/1/b4333c1b66bd48049fcb7272daa8e4e3/arti...
189,mlruns/1/b4333c1b66bd48049fcb7272daa8e4e3/arti...


In [13]:
# Detener el servidor y consolidar SQLite antes de comprimir el respaldo.
if 'servidor' in globals() and servidor.poll() is None:
    servidor.terminate()
    servidor.wait(timeout=30)
import sqlite3
import shutil
with sqlite3.connect('mlflow.db') as conexion:
    conexion.execute('PRAGMA wal_checkpoint(TRUNCATE)')
zip_path = shutil.make_archive('/content/C5061_Entrega_JimenezMarco', 'zip', root_dir=PROYECTO)
files.download(zip_path)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>